<div dir="rtl" align="right">

# رگرسیون Ridge (منظم‌سازی L2)

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
در بیش‌برازش، ضرایب مدل خیلی بزرگ می‌شوند (مثلاً ۸۰۰۰+ و ۷۹۹۰−) تا منحنی از همه نقاط بگذرد. Ridge به تابع هزینه یک **جریمه** برای ضرایب بزرگ اضافه می‌کند:

$$J(\mathbf w)=\underbrace{\sum_i(y_i-\hat y_i)^2}_{\text{error}}+\alpha\underbrace{\sum_j w_j^2}_{\text{L2 penalty}}$$

- $\alpha=0$: همان رگرسیون خطی معمولی.
- $\alpha$ بزرگ: ضرایب به سمت صفر کوچک می‌شوند و منحنی صاف‌تر می‌شود.
- $\alpha$ خیلی بزرگ: همه ضرایب تقریباً صفر؛ مدل فقط میانگین را می‌گوید (کم‌برازش).

> جریمه روی $b$ (عرض از مبدأ) اعمال نمی‌شود.

## فرمول بسته
$$\mathbf w^*=(X^\top X+\alpha I)^{-1}X^\top\mathbf y$$

اضافه شدن $\alpha I$ ماتریس را همیشه وارون‌پذیر می‌کند. برای همین Ridge درمان خوبی برای **هم‌خطی** هم هست.

## مسیر ضرایب
وقتی $\alpha$ را زیاد می‌کنیم، همه ضرایب **به‌آرامی** کوچک می‌شوند ولی هیچ‌کدام دقیقاً صفر نمی‌شود:

> [مسیر ضرایب Ridge روی داده دیابت.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/ridge_path.png)

*مسیر ضرایب Ridge روی داده دیابت.*

## آزمایشگاه
چندجمله‌ای درجه ۱۲ را با $\alpha$ های مختلف ببین. وقتی $\alpha$ خیلی کوچک است منحنی دیوانه‌وار نوسان می‌کند و ضرایب (نمودار میله‌ای) بزرگ‌اند:

[[lab:reg_ridge]]

## مقیاس‌بندی واجب است
جریمه $\sum w_j^2$ با همه ضرایب یکسان رفتار می‌کند. اگر یک ویژگی بر حسب متر و دیگری بر حسب میلی‌متر باشد، جریمه عادلانه نیست. پس همیشه قبل از Ridge از `StandardScaler` استفاده کن.



</div>

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(1)
x = rng.uniform(0, 1, 20)[:, None]
y = np.sin(2 * np.pi * x.ravel()) + rng.normal(0, 0.2, 20)

for name, reg in [("Linear", LinearRegression()), ("Ridge a=0.001", Ridge(0.001)), ("Ridge a=1", Ridge(1)), ("Ridge a=100", Ridge(100))]:
    m = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), reg).fit(x, y)
    w = m[-1].coef_
    print(f"{name:14s} largest |w| = {abs(w).max():12.2f}   sum of w^2 = {np.sum(w ** 2):14.2f}")

<div dir="rtl" align="right">



## انتخاب α با اعتبارسنجی
`RidgeCV` چند مقدار α را با اعتبارسنجی متقاطع امتحان می‌کند:



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30))).fit(X_tr, y_tr)
print("Chosen alpha:", round(model[-1].alpha_, 3))
print("Test R2:", round(model.score(X_te, y_te), 3))

<div dir="rtl" align="right">



| هایپرپارامتر | اثر | مقدار معمول برای شروع |
|---|---|---|
| `alpha` | قدرت جریمه | 0.1 تا 100، روی مقیاس لگاریتمی جستجو کن |
| `solver` | روش حل | `auto` |

## مقایسه با رگرسیون خطی
| | خطی | Ridge |
|---|---|---|
| بیش‌برازش با ویژگی زیاد | زیاد | کمتر |
| هم‌خطی | ضرایب ناپایدار | ضرایب پایدار |
| انتخاب ویژگی | ندارد | ندارد (ضرایب صفر نمی‌شوند) |
| نیاز به مقیاس‌بندی | خیر | **بله** |

</div>

<div dir="rtl" align="right">

### تمرین

Ridge را با فرمول بسته (X^T X + αI)^-1 X^T y روی داده دیابت مقیاس‌شده پیاده کن و با sklearn مقایسه کن.

</div>

In [ ]:
# Your answer here